In [1]:
#!/usr/bin/env python
"""
════════════════════════════════════════════════════════════════════════
 SWINIR 2× SR + DENOISING — SUBMISSION NOTEBOOK
════════════════════════════════════════════════════════════════════════
 All tunables and paths are in SECTION 1.
 Everything else runs automatically.
════════════════════════════════════════════════════════════════════════
"""

'\n════════════════════════════════════════════════════════════════════════\n SWINIR 2× SR + DENOISING — SUBMISSION NOTEBOOK\n════════════════════════════════════════════════════════════════════════\n All tunables and paths are in SECTION 1.\n Everything else runs automatically.\n════════════════════════════════════════════════════════════════════════\n'

In [2]:
#
#  ★ DIRECTORIES — update these to match your Kaggle inputs
#  ★ TUNABLES    — tweak these right before submitting
#

# ── Directories ──
COMP_DATA    = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK"
TEST_LR_DIR  = COMP_DATA + "/test/NoisyLR"
TRAIN_GT_DIR = COMP_DATA + "/train/GT"          # only used for validation cell
TRAIN_LR_DIR = COMP_DATA + "/train/NoisyLR"     # only used for validation cell

# Weights — point to your uploaded .pth file
# Options: best_model.pth, best_ema.pth, final_model.pth, final_ema.pth
WEIGHTS_PATH = "/kaggle/input/datasets/khxjamohammed/swinir-phase1-mid/checkpoints/best_ema.pth"

# ── Inference Tunables ──
USE_TTA           = True     # 8× self-ensemble (flip + rotate). Slower but +0.1-0.3 dB
TTA_MODE          = "full"   # "full" = 8× (4 rot × 2 flip),  "light" = 4× (flips only)
CLAMP_MIN         = 0.0      # output pixel min (competition requires [0, 1])
CLAMP_MAX         = 1.0      # output pixel max
GEOMETRIC_MEAN    = False    # True = geometric mean of TTA preds (can help SSIM), False = arithmetic
OVERLAP_BLEND     = False    # True = split large images into overlapping tiles (not needed for 128→256)

# ── Model Config (must match training exactly — don't change unless you trained differently) ──
MODEL_EMBED_DIM    = 180
MODEL_DEPTHS       = [6, 6, 6, 6, 6, 6]
MODEL_NUM_HEADS    = [6, 6, 6, 6, 6, 6]
MODEL_WINDOW_SIZE  = 8
MODEL_MLP_RATIO    = 2
MODEL_UPSCALE      = 2
MODEL_NUM_FEAT     = 64
MODEL_IMG_RANGE    = 1.0
MODEL_RESI_CONN    = '1conv'

# ── Output ──
SUB_DIR = "/kaggle/working/submission"
CSV_PATH = "/kaggle/working/submission.csv"

In [3]:

import os, glob, math, time, random, warnings
import numpy as np
import pandas as pd
import base64
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
os.makedirs(SUB_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Test images: {len(os.listdir(TEST_LR_DIR))}")
print(f"Weights: {WEIGHTS_PATH}")
print(f"TTA: {TTA_MODE if USE_TTA else 'OFF'}")

Device: cuda
Test images: 200
Weights: /kaggle/input/datasets/khxjamohammed/swinir-phase1-mid/checkpoints/best_ema.pth
TTA: full


In [4]:
# Must match training code exactly.

def drop_path_fn(x, drop_prob=0., training=False):
    if drop_prob == 0. or not training:
        return x
    keep = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    mask = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
    mask.floor_()
    return x.div(keep) * mask

class DropPath(nn.Module):
    def __init__(self, p=None):
        super().__init__()
        self.p = p
    def forward(self, x):
        return drop_path_fn(x, self.p, self.training)

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)

def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None,
                 attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_flat = torch.flatten(coords, 1)
        rel = coords_flat[:, :, None] - coords_flat[:, None, :]
        rel = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += self.window_size[0] - 1
        rel[:, :, 1] += self.window_size[1] - 1
        rel[:, :, 0] *= 2 * self.window_size[1] - 1
        self.register_buffer("relative_position_index", rel.sum(-1))
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        rpb = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)
        attn = attn + rpb.permute(2, 0, 1).contiguous().unsqueeze(0)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))

class SwinTransformerBlock(nn.Module):
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop=0., attn_drop=0., drop_path=0., norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(input_resolution) <= self.window_size:
            self.shift_size = 0
            self.window_size = min(input_resolution)
        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=(self.window_size, self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)

    def _compute_mask(self, H, W, device):
        if self.shift_size <= 0:
            return None
        img_mask = torch.zeros((1, H, W, 1), device=device)
        slices_h = [slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None)]
        slices_w = [slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None)]
        cnt = 0
        for h in slices_h:
            for w in slices_w:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        mask_win = window_partition(img_mask, self.window_size)
        mask_win = mask_win.view(-1, self.window_size * self.window_size)
        attn_mask = mask_win.unsqueeze(1) - mask_win.unsqueeze(2)
        return attn_mask.masked_fill(attn_mask != 0, -100.0).masked_fill(attn_mask == 0, 0.0)

    def forward(self, x, x_size):
        H, W = x_size
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x).view(B, H, W, C)
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x
        x_win = window_partition(shifted_x, self.window_size)
        x_win = x_win.view(-1, self.window_size * self.window_size, C)
        attn_mask = self._compute_mask(H, W, x.device)
        attn_win = self.attn(x_win, mask=attn_mask)
        attn_win = attn_win.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_win, self.window_size, H, W)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=1, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Identity() if patch_size == 1 else nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = norm_layer(embed_dim) if norm_layer else None
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        if self.norm:
            x = self.norm(x)
        return x

class PatchUnEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=1, in_chans=3, embed_dim=96):
        super().__init__()
        self.embed_dim = embed_dim
    def forward(self, x, x_size):
        return x.transpose(1, 2).view(x.shape[0], self.embed_dim, x_size[0], x_size[1])

class Upsample(nn.Sequential):
    def __init__(self, scale, num_feat):
        m = []
        for _ in range(int(math.log(scale, 2))):
            m.append(nn.Conv2d(num_feat, 4 * num_feat, 3, 1, 1))
            m.append(nn.PixelShuffle(2))
        super().__init__(*m)

class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(
                dim=dim, input_resolution=input_resolution, num_heads=num_heads,
                window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop, attn_drop=attn_drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                norm_layer=norm_layer)
            for i in range(depth)])
    def forward(self, x, x_size):
        for blk in self.blocks:
            x = blk(x, x_size)
        return x

class RSTB(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.LayerNorm, resi_connection='1conv', **kw):
        super().__init__()
        self.residual_group = BasicLayer(
            dim=dim, input_resolution=input_resolution, depth=depth,
            num_heads=num_heads, window_size=window_size, mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop, attn_drop=attn_drop,
            drop_path=drop_path, norm_layer=norm_layer)
        if resi_connection == '1conv':
            self.conv = nn.Conv2d(dim, dim, 3, 1, 1)
        elif resi_connection == '3conv':
            self.conv = nn.Sequential(
                nn.Conv2d(dim, dim // 4, 3, 1, 1), nn.LeakyReLU(0.2, True),
                nn.Conv2d(dim // 4, dim // 4, 1, 1, 0), nn.LeakyReLU(0.2, True),
                nn.Conv2d(dim // 4, dim, 3, 1, 1))
        self.patch_embed = PatchEmbed(img_size=input_resolution[0], embed_dim=dim)
        self.patch_unembed = PatchUnEmbed(img_size=input_resolution[0], embed_dim=dim)
    def forward(self, x, x_size):
        return self.patch_embed(
            self.conv(self.patch_unembed(self.residual_group(x, x_size), x_size))) + x

class SwinIR(nn.Module):
    def __init__(self, img_size=128, patch_size=1, in_chans=1, embed_dim=180,
                 depths=(6,6,6,6,6,6), num_heads=(6,6,6,6,6,6), window_size=8,
                 mlp_ratio=2., qkv_bias=True, qk_scale=None, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0.1, norm_layer=nn.LayerNorm,
                 upscale=2, img_range=1., upsampler='pixelshuffle',
                 resi_connection='1conv', num_feat=64, **kw):
        super().__init__()
        self.img_range = img_range
        self.upscale = upscale
        self.upsampler = upsampler
        self.window_size = window_size
        if in_chans == 3:
            self.mean = torch.Tensor([0.4488, 0.4371, 0.4040]).view(1, 3, 1, 1)
        else:
            self.mean = torch.zeros(1, 1, 1, 1)
        self.conv_first = nn.Conv2d(in_chans, embed_dim, 3, 1, 1)
        patches_resolution = [img_size // patch_size, img_size // patch_size]
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size,
                                      in_chans=embed_dim, embed_dim=embed_dim)
        self.patch_unembed = PatchUnEmbed(img_size=img_size, patch_size=patch_size,
                                          in_chans=embed_dim, embed_dim=embed_dim)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        self.layers = nn.ModuleList()
        for i_layer in range(len(depths)):
            layer = RSTB(
                dim=embed_dim, input_resolution=(patches_resolution[0], patches_resolution[1]),
                depth=depths[i_layer], num_heads=num_heads[i_layer],
                window_size=window_size, mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias, qk_scale=qk_scale, drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                norm_layer=norm_layer, resi_connection=resi_connection)
            self.layers.append(layer)
        self.norm = norm_layer(embed_dim)
        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, 1, 1)
        if self.upsampler == 'pixelshuffle':
            self.conv_before_upsample = nn.Sequential(
                nn.Conv2d(embed_dim, num_feat, 3, 1, 1), nn.LeakyReLU(inplace=True))
            self.upsample = Upsample(upscale, num_feat)
            self.conv_last = nn.Conv2d(num_feat, in_chans, 3, 1, 1)
        elif self.upsampler == '':
            self.conv_last = nn.Conv2d(embed_dim, in_chans, 3, 1, 1)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def check_image_size(self, x):
        _, _, h, w = x.size()
        ws = self.window_size
        pad_h = (ws - h % ws) % ws
        pad_w = (ws - w % ws) % ws
        return F.pad(x, [0, pad_w, 0, pad_h], mode='reflect') if pad_h + pad_w > 0 else x

    def forward_features(self, x, x_size):
        x = self.patch_embed(x)
        for layer in self.layers:
            x = layer(x, x_size)
        x = self.norm(x)
        x = self.patch_unembed(x, x_size)
        return x

    def forward(self, x):
        H, W = x.shape[2:]
        x = self.check_image_size(x)
        self.mean = self.mean.type_as(x)
        x = (x - self.mean) * self.img_range
        if self.upsampler == 'pixelshuffle':
            x_first = self.conv_first(x)
            res = self.conv_after_body(self.forward_features(x_first, (x.shape[2], x.shape[3]))) + x_first
            x = self.conv_before_upsample(res)
            x = self.conv_last(self.upsample(x))
        elif self.upsampler == '':
            x_first = self.conv_first(x)
            res = self.conv_after_body(self.forward_features(x_first, (x.shape[2], x.shape[3]))) + x_first
            x = x + self.conv_last(res)
        x = x / self.img_range + self.mean
        return x[:, :, :H * self.upscale, :W * self.upscale]

In [5]:

model = SwinIR(
    img_size=128, patch_size=1, in_chans=1,
    embed_dim=MODEL_EMBED_DIM, depths=MODEL_DEPTHS, num_heads=MODEL_NUM_HEADS,
    window_size=MODEL_WINDOW_SIZE, mlp_ratio=MODEL_MLP_RATIO,
    upscale=MODEL_UPSCALE, img_range=MODEL_IMG_RANGE,
    upsampler='pixelshuffle', resi_connection=MODEL_RESI_CONN,
    num_feat=MODEL_NUM_FEAT, drop_path_rate=0.0,  # no drop path at inference
).to(DEVICE)

# Load weights
state = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=True)
# Handle wrapped state dicts
if 'model' in state:
    state = state['model']
elif 'params' in state:
    state = state['params']
elif 'params_ema' in state:
    state = state['params_ema']

model.load_state_dict(state, strict=False)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} parameters")

Model loaded: 11,747,733 parameters


In [6]:

@torch.no_grad()
def predict_single(model, lr_np):
    """
    Run inference on a single image with optional TTA.
    Input:  lr_np — numpy (128, 128) float32
    Output: pred  — numpy (256, 256) float32
    """
    lr_t = torch.from_numpy(lr_np).float().unsqueeze(0).unsqueeze(0)  # (1,1,128,128)

    if not USE_TTA:
        pred = model(lr_t.to(DEVICE)).cpu()
        return pred.squeeze().numpy()

    # Self-ensemble TTA
    preds = []

    if TTA_MODE == "full":
        # 8×: 4 rotations × 2 flips
        transforms = [(flip, k) for flip in (False, True) for k in range(4)]
    else:
        # 4×: flips only (horizontal, vertical, both, none)
        transforms = [(False, 0), (True, 0), (False, 2), (True, 2)]

    for do_flip, k in transforms:
        x = lr_t.clone()
        if do_flip:
            x = torch.flip(x, [-1])
        x = torch.rot90(x, k, [-2, -1])

        p = model(x.to(DEVICE)).cpu()

        # Undo transforms
        p = torch.rot90(p, -k, [-2, -1])
        if do_flip:
            p = torch.flip(p, [-1])
        preds.append(p)

    stacked = torch.stack(preds)

    if GEOMETRIC_MEAN:
        # Geometric mean — can preserve sharpness better for SSIM
        stacked = stacked.clamp(min=1e-8)
        result = torch.exp(torch.log(stacked).mean(0))
    else:
        result = stacked.mean(0)

    return result.squeeze().numpy()

In [7]:

def validate_on_train():
    """Quick PSNR + SSIM check on training pairs. Run this to verify quality."""
    from skimage.metrics import peak_signal_noise_ratio as psnr_fn
    from skimage.metrics import structural_similarity as ssim_fn

    gt_files = sorted(os.listdir(TRAIN_GT_DIR))
    lr_files = sorted(os.listdir(TRAIN_LR_DIR))

    # Use last 120 as pseudo-validation
    n_val = min(120, len(gt_files))
    indices = list(range(len(gt_files) - n_val, len(gt_files)))

    psnr_sum, ssim_sum, count = 0.0, 0.0, 0

    for i in indices:
        lr = np.load(os.path.join(TRAIN_LR_DIR, lr_files[i])).astype(np.float32)
        gt = np.load(os.path.join(TRAIN_GT_DIR, gt_files[i])).astype(np.float32)

        pred = predict_single(model, lr)
        pred = np.clip(pred, CLAMP_MIN, CLAMP_MAX).astype(np.float32)

        dr = gt.max() - gt.min()
        if dr > 1e-8:
            psnr_sum += psnr_fn(gt, pred, data_range=dr)
            ssim_sum += ssim_fn(gt, pred, data_range=dr)
            count += 1

        if (count) % 30 == 0:
            print(f"  {count}/{n_val}  running PSNR={psnr_sum/count:.3f}  SSIM={ssim_sum/count:.4f}")

    avg_psnr = psnr_sum / max(count, 1)
    avg_ssim = ssim_sum / max(count, 1)
    print(f"\n  ✓ Validation ({count} images):")
    print(f"    PSNR: {avg_psnr:.3f} dB")
    print(f"    SSIM: {avg_ssim:.4f}")
    print(f"    TTA:  {TTA_MODE if USE_TTA else 'OFF'}")
    return avg_psnr, avg_ssim

# Uncomment the line below to run validation before generating submission:
#validate_on_train()

In [8]:

print("\n" + "=" * 60)
print("  GENERATING PREDICTIONS")
print("=" * 60)

test_files = sorted([f for f in os.listdir(TEST_LR_DIR) if f.endswith(".npy")])
print(f"  Test images: {len(test_files)}")
print(f"  TTA: {TTA_MODE if USE_TTA else 'OFF'}")
print(f"  Clamp: [{CLAMP_MIN}, {CLAMP_MAX}]")

t0 = time.time()
n_total = len(test_files)

for i, fname in enumerate(test_files):
    lr = np.load(os.path.join(TEST_LR_DIR, fname)).astype(np.float32)

    pred = predict_single(model, lr)

    # Post-process
    pred = np.clip(pred, CLAMP_MIN, CLAMP_MAX).astype(np.float32)

    # Safety checks
    assert pred.shape == (256, 256), f"Shape error: {pred.shape}"
    assert pred.dtype == np.float32, f"Dtype error: {pred.dtype}"
    assert not np.isnan(pred).any(), f"NaN in {fname}"
    assert not np.isinf(pred).any(), f"Inf in {fname}"

    np.save(os.path.join(SUB_DIR, fname), pred)

    if (i + 1) % 50 == 0 or (i + 1) == n_total:
        elapsed = time.time() - t0
        speed = (i + 1) / elapsed
        eta = (n_total - i - 1) / max(speed, 0.01)
        print(f"  {i+1:4d}/{n_total}  ({speed:.1f} img/s, ETA {eta:.0f}s)")

elapsed = time.time() - t0
print(f"\n  Done: {n_total} images in {elapsed:.1f}s ({n_total/elapsed:.1f} img/s)")


  GENERATING PREDICTIONS
  Test images: 200
  TTA: full
  Clamp: [0.0, 1.0]
    50/200  (0.5 img/s, ETA 284s)
   100/200  (0.5 img/s, ETA 196s)
   150/200  (0.5 img/s, ETA 99s)
   200/200  (0.5 img/s, ETA 0s)

  Done: 200 images in 398.3s (0.5 img/s)


In [9]:

print("\n" + "=" * 60)
print("  CREATING SUBMISSION CSV")
print("=" * 60)

submission_dir = SUB_DIR
rows = []
files = sorted([f for f in os.listdir(submission_dir) if f.endswith(".npy")])

for idx, file in enumerate(files, start=1):
    path = os.path.join(submission_dir, file)
    arr = np.load(path)

    buffer = BytesIO()
    np.save(buffer, arr)
    encoded = base64.b64encode(buffer.getvalue()).decode()

    rows.append({
        "id": idx,
        "npy_base64": encoded
    })

df = pd.DataFrame(rows)
df.to_csv(CSV_PATH, index=False)

print(f"  Rows: {len(df)}")
print(f"  CSV: {CSV_PATH} ({os.path.getsize(CSV_PATH) / 1e6:.1f} MB)")
print(df.head())


  CREATING SUBMISSION CSV
  Rows: 200
  CSV: /kaggle/working/submission.csv (69.9 MB)
   id                                         npy_base64
0   1  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
1   2  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
2   3  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
3   4  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
4   5  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...


In [10]:

print("\n" + "=" * 60)
print("  VERIFICATION")
print("=" * 60)

test_count = len(test_files)
sub_count = len(files)
csv_count = len(df)

print(f"  Test images:      {test_count}")
print(f"  Prediction files: {sub_count}")
print(f"  CSV rows:         {csv_count}")

assert sub_count == test_count, f"MISSING! {sub_count} vs {test_count}"
assert csv_count == test_count, f"CSV MISMATCH! {csv_count} vs {test_count}"

# Spot check
random_file = random.choice(files)
arr = np.load(os.path.join(SUB_DIR, random_file))
print(f"\n  Spot check ({random_file}):")
print(f"    Shape: {arr.shape}")
print(f"    Dtype: {arr.dtype}")
print(f"    Range: [{arr.min():.4f}, {arr.max():.4f}]")
print(f"    Mean:  {arr.mean():.4f}")

print(f"\n  ✓ ALL CHECKS PASSED — READY TO SUBMIT")
print("=" * 60)


  VERIFICATION
  Test images:      200
  Prediction files: 200
  CSV rows:         200

  Spot check (000148.npy):
    Shape: (256, 256)
    Dtype: float32
    Range: [0.0000, 1.0000]
    Mean:  0.4314

  ✓ ALL CHECKS PASSED — READY TO SUBMIT
